In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

INPUT_PATH   = Path("input_datasets")
SENSOR_COUNT = 2

OUTPUT_PATH  = Path("merged_datasets")

PEOPLE = ["Hiruni", "Mineth", "Tanushka", "Thinula", "Thanushka", "Sineth"]

ACTIVITIES = ["Bending", "Idle", "Picking", "Pushing"]

ACTIVITY_LABELS = {
    "Bending": 0,
    "Idle": 1,
    "Picking": 2,
    "Pushing": 3
}

SENSOR_POSITIONS = [
    "D_Leg",
    "D_Upper_Arm",
    "L_Wrist",
    "R_Wrist",
    "D_Upper_Arm"
]

ACCELEROMETER_COLUMNS = {
    "FreeAcc_X": "Acc_X",
    "FreeAcc_Y": "Acc_Y",
    "FreeAcc_Z": "Acc_Z"
}


def get_sensor_key(file_path: Path) -> str:
    file_stem = file_path.stem

    for sensor_position in SENSOR_POSITIONS:
        if (file_stem == sensor_position or file_stem.startswith(f"{sensor_position}_")):
            return sensor_position

    raise ValueError(f"Unknown sensor placement: {file_path.name}")

def read_sensor_file(
    file_path: Path,
    sensor_key: str
) -> pd.DataFrame:

    df = pd.read_csv(
        file_path,
        skiprows=11,
        usecols=[
            "SampleTimeFine",
            *ACCELEROMETER_COLUMNS.keys()
        ]
    )

    renamed_columns = {
        original_name: f"{sensor_key}_{new_name}"
        for original_name, new_name
        in ACCELEROMETER_COLUMNS.items()
    }

    return df.rename(columns=renamed_columns)  


OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

summary = {
    activity: {
        "People": set(),
        "Collections": {}
    }
    for activity in ACTIVITIES
}

for person in PEOPLE:
    person_path = INPUT_PATH / person

    if not person_path.exists():
        print(f"Person folder not found: {person_path.resolve()}")
        continue

    person_data = []

    for activity in ACTIVITIES:
        activity_folders = sorted([
            folder
            for folder in person_path.glob(f"{person}_{activity}*")
            if folder.is_dir()
        ])

        if not activity_folders:
            print
            continue

        for activity_folder in activity_folders:
            sensor_files = sorted(
                activity_folder.glob("*.csv")
            )

            # Check the exact number of sensors
            if len(sensor_files) != SENSOR_COUNT:
                raise ValueError(
                    f"{activity_folder} contains only "
                    f"{len(sensor_files)} sensor files. "
                    f"Expected {SENSOR_COUNT} sensor files."
                )

            sensor_data = {}

            for sensor_file in sensor_files:
                sensor_key = get_sensor_key(sensor_file)
                sensor_df = read_sensor_file(
                    sensor_file,
                    sensor_key
                )

                sensor_data[sensor_key] = sensor_df

            # Check whether every sensor has the same row count
            sensor_frames = list(sensor_data.values())

            trial_df = sensor_frames[0]

            for sensor_df in sensor_frames[1:]:
                trial_df = trial_df.merge(
                    sensor_df,
                    on="SampleTimeFine",
                    how="inner",
                    validate="one_to_one"
                )

            trial_df = (
                trial_df
                .sort_values("SampleTimeFine")
                .reset_index(drop=True)
            )
            
            if trial_df.empty:
                raise ValueError(
                    f"No matching SampleTimeFine values found in "
                    f"{activity_folder}"
                )
            
            original_row_counts = {
                sensor_key: len(sensor_df)
                for sensor_key, sensor_df in sensor_data.items()
            }

            discarded_rows = {
                sensor_key: row_count - len(trial_df)
                for sensor_key, row_count
                in original_row_counts.items()
            }

            if any(count > 0 for count in discarded_rows.values()):
                print(
                    f"Warning: unmatched samples removed from "
                    f"{activity_folder.name}: {discarded_rows}"
                )
            
            

            # Add metadata 
            trial_df["Activity"] = ACTIVITY_LABELS[activity]

            person_data.append(trial_df)
            
            # Record this successfully processed collection
            summary[activity]["People"].add(person)
            summary[activity]["Collections"].setdefault(person,[])
            summary[activity]["Collections"][person].append(activity_folder.name)

    # Vertically combine this person's activity trials
    if person_data:
        person_df = pd.concat(person_data, ignore_index=True)
        person_df.to_csv( OUTPUT_PATH / f"{person}.csv", index=False)


summary_rows = []

for activity in ACTIVITIES:
    activity_summary = summary[activity]
    collections_by_person = []

    for person in sorted(activity_summary["Collections"]):
        collection_folders = sorted(
            activity_summary["Collections"][person]
        )

        indexed_collections = ", ".join(
            f"{index}. {folder_name}"
            for index, folder_name
            in enumerate(collection_folders, start=1)
        )

        collections_by_person.append(
            f"{person}: {indexed_collections}"
        )

    total_collections = sum(
        len(collection_folders)
        for collection_folders
        in activity_summary["Collections"].values()
    )

    summary_rows.append({
        "Activity": activity,
        "Number of People": len(activity_summary["People"]),
        "People": ", ".join(
            sorted(activity_summary["People"])
        ),
        "Number of Collections": total_collections,
        "Collections by Person": " | ".join(
            collections_by_person
        )
    })

summary_df = pd.DataFrame(summary_rows)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(summary_df)

Person folder not found: C:\Users\Sina\Documents\University\Swinburne\COS40005\Code\wearable-ai-iot-berry-picking\ml\my_merger\input_datasets\Hiruni
Person folder not found: C:\Users\Sina\Documents\University\Swinburne\COS40005\Code\wearable-ai-iot-berry-picking\ml\my_merger\input_datasets\Mineth
Person folder not found: C:\Users\Sina\Documents\University\Swinburne\COS40005\Code\wearable-ai-iot-berry-picking\ml\my_merger\input_datasets\Tanushka
Person folder not found: C:\Users\Sina\Documents\University\Swinburne\COS40005\Code\wearable-ai-iot-berry-picking\ml\my_merger\input_datasets\Thinula
Person folder not found: C:\Users\Sina\Documents\University\Swinburne\COS40005\Code\wearable-ai-iot-berry-picking\ml\my_merger\input_datasets\Thanushka


,Activity,Number of People,People,Number of Collections,Collections by Person
0,Bending,1,Sineth,7,"Sineth: 1. Sineth_Bending_1, 2. Sineth_Bending_2, 3. Sineth_Bending_3, 4. Sineth_Bending_4, 5. Sineth_Bending_5, 6. Sineth_Bending_6, 7. Sineth_Bending_7"
1,Idle,0,,0,
2,Picking,0,,0,
3,Pushing,0,,0,
